# Tutorial 2: Configuration & Tuning

How to control which methods run, adjust weights, tune performance, and use the FeatureStore cache.

In [ ]:
import numpy as np
import time
from the_similarity import load, search, project, Config, FeatureStore

In [ ]:
# Generate synthetic data
np.random.seed(42)
n = 2000
prices = 100 * np.exp(np.cumsum(np.random.normal(0.0002, 0.01, n)))
history = load(prices)
query = load(prices[-60:])

## 1. Method Selection

Control which methods run via `active_methods`. Fewer methods = faster but less accurate.

In [ ]:
# All 9 methods (default) — most accurate, ~1s on 2000 bars
cfg_full = Config()  
print(f"Default active methods: {len(cfg_full.active_methods)}")
print(f"Methods: {cfg_full.active_methods}")

In [ ]:
# DTW + Pearson only — fastest, skip all Tier 2
cfg_fast = Config(
    active_methods=["dtw", "pearson_warped"],
    tier2_candidates=0,  # skip Tier 2 entirely
)

t0 = time.time()
results_fast = search(query, history, config=cfg_fast, stride=3)
print(f"Fast mode: {time.time() - t0:.2f}s, best score: {results_fast.best.confidence_score:.1f}")

In [ ]:
# Shape + dynamics only — good middle ground
cfg_mid = Config(
    active_methods=["dtw", "pearson_warped", "koopman", "wavelet_spectrum"],
)

t0 = time.time()
results_mid = search(query, history, config=cfg_mid, stride=3)
print(f"Mid mode: {time.time() - t0:.2f}s, best score: {results_mid.best.confidence_score:.1f}")

## 2. Custom Weights

Weights are automatically renormalized across active methods. You can emphasize what matters to you.

In [ ]:
# Emphasize dynamical similarity over shape
results_dynamics = search(
    query, history,
    weights={
        "koopman": 0.40,
        "wavelet_spectrum": 0.25,
        "bempedelis_r2": 0.20,
        "dtw": 0.05,
        "pearson_warped": 0.05,
        "bempedelis_smoothness": 0.05,
    },
    stride=3,
)
print(f"Dynamics-weighted best score: {results_dynamics.best.confidence_score:.1f}")

## 3. Performance Tuning

Key knobs for speed vs. accuracy tradeoff.

In [ ]:
# stride: how many bars to skip between candidate windows
# stride=1 (default): check every position — most thorough
# stride=3: 3x faster, might miss patterns at non-aligned positions
# stride=5: 5x faster, good for exploration

for stride in [1, 3, 5]:
    t0 = time.time()
    r = search(query, history, stride=stride)
    elapsed = time.time() - t0
    print(f"stride={stride}: {elapsed:.2f}s, best={r.best.confidence_score:.1f}")

In [ ]:
# tier1_candidates: how many survive the prefilter
# Lower = faster Tier 1 scoring, but might miss good matches

for t1 in [100, 500, 1000]:
    cfg = Config(tier1_candidates=t1)
    t0 = time.time()
    r = search(query, history, config=cfg, stride=3)
    elapsed = time.time() - t0
    print(f"tier1={t1}: {elapsed:.2f}s, best={r.best.confidence_score:.1f}")

## 4. FeatureStore Caching

Cache expensive Tier 2 computations in SQLite. Especially useful for backtesting where the same windows are scored repeatedly.

In [ ]:
import tempfile, os

db_path = os.path.join(tempfile.gettempdir(), "tutorial_cache.db")
store = FeatureStore(db_path)

# First search — populates cache
t0 = time.time()
r1 = search(query, history, feature_store=store, stride=3)
print(f"First search: {time.time() - t0:.2f}s, cache size: {store.size}")

# Second search — cache hits
t0 = time.time()
r2 = search(query, history, feature_store=store, stride=3)
print(f"Second search: {time.time() - t0:.2f}s, cache size: {store.size}")

# Clean up
store.clear()
os.unlink(db_path)

## 5. Forecast Cone Tuning

In [ ]:
results = search(query, history, stride=3)

# Default: flat confidence
f1 = project(results, history, forward_bars=50)
print(f"Default P50 endpoint: {f1.curves[50][-1]:+.4f}")
print(f"Default P10-P90 spread: {f1.curves[90][-1] - f1.curves[10][-1]:.4f}")

# With confidence decay: cone widens over time
cfg_decay = Config(confidence_decay_rate=0.03)
f2 = project(results, history, forward_bars=50, config=cfg_decay)
print(f"\nWith decay P10-P90 spread: {f2.curves[90][-1] - f2.curves[10][-1]:.4f}")

# With Koopman blend: physics-informed P50
cfg_koop = Config(koopman_blend_weight=0.3)
f3 = project(results, history, forward_bars=50, config=cfg_koop)
print(f"\nKoopman-blended P50 endpoint: {f3.curves[50][-1]:+.4f}")
if f3.koopman_forecast:
    print(f"Koopman trajectory endpoint: {f3.koopman_forecast.trajectory[-1]:+.4f}")